In [ ]:

# =====================================================================
# CropPulse - Operational crop vegetation monitoring (Phase 1)
# Sentinel-2 (optical) + Sentinel-1 (radar) + ERA5 (weather), Lower Saxony
# Centerpiece: use S1 SAR to fill Sentinel-2 cloud gaps in the NDVI/LAI curve
# Run in Google Colab cell-by-cell. Inspired by operational ag-monitoring
# services (Daily LAI / Drought / Cultivation Patterns).
# =====================================================================

# ---- CELL 0: install + auth + init -----------------------------------
# !pip install earthengine-api geemap eemont scikit-learn pandas matplotlib --quiet
import ee, geemap, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict
ee.Authenticate()           # run once, first time
ee.Initialize(project="ee-sanjusajimon76")
print("EE initialized")

In [ ]:

# ---- CELL 1: config --------------------------------------------------
EUROCROPS_ASSET = "projects/ee-sanjusajimon76/assets/eurocrops_nds"
YEAR            = 2021                      # MATCH to EuroCrops declaration year
START, END      = f"{YEAR}-01-01", f"{YEAR}-12-31"
N_PER_CROP    = 30
CROP_KEYWORDS = {
    "winter_wheat":  ["winter_common_soft_wheat"],
    "sugar_beet":    ["sugar_beet"],
    "grassland":     ["pasture_meadow_grassland_grass"],
    "winter_barley": ["winter_barley"],
    "maize":         ["grain_maize_corn_popcorn", "green_silo_maize"],  # grain+silage
    "winter_rape":   ["winter_rapeseed_rape"],
    "potatoes":      ["potatoes"],
}
# small AOI rectangle in Lower Saxony cropland (edit after first look);
# default ~ Hildesheim Boerde, a fertile arable area
AOI = ee.Geometry.Rectangle([9.85, 52.05, 10.15, 52.25])

In [ ]:

# ---- CELL 2: load EuroCrops, inspect, filter to our crops ------------
ec = ee.FeatureCollection(EUROCROPS_ASSET).filterBounds(AOI)
print("Sample feature properties (find the crop-name field, e.g. EC_hcat_n):")
print(ec.first().toDictionary().getInfo())     # <-- read the attribute keys here
CROP_FIELD = "EC_hcat_n"                         # adjust if the printout differs


def pick(names):
    f = ee.Filter.inList(CROP_FIELD, names)
    return ee.FeatureCollection(ec.filter(f)).randomColumn("r", 42).limit(N_PER_CROP, "r")

fields = {c: pick(kw) for c, kw in CROP_KEYWORDS.items()}
for c, fc in fields.items():
    print(c, "fields:", fc.size().getInfo())
# tag each field with crop + a stable id, merge
def tag(fc, crop):
    return fc.map(lambda ft: ft.set({"crop": crop}))
fc_list = [tag(fields[c], c) for c in CROP_KEYWORDS]
fields_all = fc_list[0]
for fc in fc_list[1:]:
    fields_all = fields_all.merge(fc)
fields_all = fields_all.map(lambda ft: ft.set("fid", ft.get("system:index")))
print("total fields:", fields_all.size().getInfo())          # expect 210
print("crops:", fields_all.aggregate_histogram("crop").getInfo())

In [ ]:

# ---- CELL 3: Sentinel-2 NDVI (+ simple LAI estimate), cloud-masked ----
csplus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
def mask_s2(img):
    cs = img.select("cs_cdf")
    return img.updateMask(cs.gte(0.6))
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(AOI).filterDate(START, END)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
        .linkCollection(csplus, ["cs_cdf"]).map(mask_s2))
def add_veg(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    # simple empirical LAI proxy (Beer-Lambert style); SNAP biophysical = rigorous route
    lai = ndvi.expression("-log((0.92 - n) / 0.78) / 0.45", {"n": ndvi}).rename("LAI")
    return img.addBands([ndvi, lai]).select(["NDVI", "LAI"]) \
              .copyProperties(img, ["system:time_start"])
s2 = s2.map(add_veg)
print("S2 scenes:", s2.size().getInfo())

In [ ]:

# ---- CELL 4: Sentinel-1 VV/VH, speckle-filtered ----------------------
def prep_s1(img):
    vv = img.select("VV"); vh = img.select("VH")
    sm = ee.Image.cat([vv, vh]).focal_mean(30, "circle", "meters")  # speckle
    ratio = sm.select("VV").subtract(sm.select("VH")).rename("VVVH")
    return sm.addBands(ratio).rename(["VV", "VH", "VVVH"]) \
             .copyProperties(img, ["system:time_start"])
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(AOI).filterDate(START, END)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .map(prep_s1))
print("S1 scenes:", s1.size().getInfo())

In [ ]:
# ---- CELL 5 (chunked): per-field time-series -> pandas ---------------
def extract(coll, bands, label):
    def per_img(img):
        d = img.date().format("YYYY-MM-dd")
        fc = img.select(bands).reduceRegions(fields_all, ee.Reducer.mean(), 20)
        return fc.map(lambda ft: ft.set({"date": d, "src": label}))
    return coll.map(per_img).flatten()

def extract_chunked(coll, bands, label):
    """Pull the per-field table one month at a time to avoid timeouts."""
    frames = []
    for m in range(1, 13):
        s = f"{YEAR}-{m:02d}-01"
        e = f"{YEAR}-{m+1:02d}-01" if m < 12 else f"{YEAR+1}-01-01"
        sub = coll.filterDate(s, e)
        if sub.size().getInfo() == 0:
            continue
        fc = extract(sub, bands, label).select(["fid","crop","date"]+bands, None, False)
        rows = fc.getInfo()["features"]
        if rows:
            frames.append(pd.DataFrame([r["properties"] for r in rows]))
        print(f"  {label} {s[:7]}: {len(rows)} rows")
    df = pd.concat(frames, ignore_index=True)
    df["date"] = pd.to_datetime(df["date"]); df["doy"] = df["date"].dt.dayofyear
    return df.dropna(subset=bands).sort_values(["fid", "date"])

print("Extracting Sentinel-2 ...")
s2_df = extract_chunked(s2, ["NDVI", "LAI"], "s2")
print("Extracting Sentinel-1 ...")
s1_df = extract_chunked(s1, ["VV", "VH", "VVVH"], "s1")
print("S2 rows:", len(s2_df), "| S1 rows:", len(s1_df))

In [ ]:

# ---- CELL 6: S1 -> NDVI gap-filling (the centerpiece) ----------------
# join S1 to nearest S2 in time per field to build a training table,
# train RF: [VV, VH, VVVH, doy, crop] -> NDVI, then predict NDVI on every S1 date
def nearest_merge(s2d, s1d, tol_days=6):
    out = []
    for fid, g2 in s2d.groupby("fid"):
        g1 = s1d[s1d.fid == fid]
        if g1.empty: continue
        m = pd.merge_asof(g1.sort_values("date"), g2.sort_values("date")[["date","NDVI","LAI"]],
                          on="date", direction="nearest", tolerance=pd.Timedelta(days=tol_days))
        m["fid"] = fid; m["crop"] = g1["crop"].iloc[0]
        out.append(m)
    return pd.concat(out, ignore_index=True)

merged = nearest_merge(s2_df, s1_df)
train = merged.dropna(subset=["NDVI"]).copy()
crop_oh = pd.get_dummies(train["crop"])
X = pd.concat([train[["VV","VH","VVVH","doy"]], crop_oh], axis=1)
y = train["NDVI"]
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42)
cv_pred = cross_val_predict(rf, X, y, cv=5)
r2 = 1 - np.sum((y-cv_pred)**2)/np.sum((y-y.mean())**2)
rf.fit(X, y)
print(f"S1->NDVI gap-fill  CV R2 = {r2:.3f}  (n={len(y)})")

# predict NDVI for ALL S1 acquisitions (dense, cloud-free) = gap-filled curve
allc = pd.get_dummies(merged["crop"])
Xall = pd.concat([merged[["VV","VH","VVVH","doy"]], allc], axis=1).reindex(columns=X.columns, fill_value=0)
merged["NDVI_s1pred"] = rf.predict(Xall)

In [ ]:
# ---- CELL 12 (step 3): stronger S1->NDVI gap-fill --------------------
# temporal lag/rolling features + gradient boosting, with HONEST per-crop
# cross-validated R2 compared against the baseline (0.769).
from sklearn.ensemble import HistGradientBoostingRegressor

# richer features on the S1 series (per field, time-ordered)
m = merged.sort_values(["fid","date"]).copy()
for col in ["VV","VH","VVVH"]:
    m[col+"_lag1"]  = m.groupby("fid")[col].shift(1)
    m[col+"_roll3"] = m.groupby("fid")[col].transform(lambda x: x.rolling(3,1).mean())
m["doy_sin"] = np.sin(2*np.pi*m["doy"]/365); m["doy_cos"] = np.cos(2*np.pi*m["doy"]/365)
FEATS = ["VV","VH","VVVH","VV_lag1","VH_lag1","VVVH_lag1",
         "VV_roll3","VH_roll3","VVVH_roll3","doy","doy_sin","doy_cos"]
m[FEATS] = m[FEATS].bfill().ffill()

def cv_r2(X, y, model, cv=5):
    p = cross_val_predict(model, X, y, cv=cv)
    return 1 - np.sum((y-p)**2)/np.sum((y-y.mean())**2), p

train = m.dropna(subset=["NDVI"]).copy()

# A) baseline (reproduce old): RF, old features
Xb = pd.concat([train[["VV","VH","VVVH","doy"]], pd.get_dummies(train["crop"])], axis=1)
r2_base,_ = cv_r2(Xb, train["NDVI"], RandomForestRegressor(300, min_samples_leaf=3, random_state=42))

# B) improved: HistGradientBoosting + lag/rolling features
Xr  = pd.concat([train[FEATS], pd.get_dummies(train["crop"])], axis=1)
hgb = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=42)
r2_rich, pred_rich = cv_r2(Xr, train["NDVI"], hgb)

print(f"baseline  (RF, old feats)      CV R2 = {r2_base:.3f}")
print(f"improved  (HGB, +lag/rolling)  CV R2 = {r2_rich:.3f}")

train = train.assign(pred=pred_rich)
print("per-crop (improved model):")
for c,g in train.groupby("crop"):
    r2c = 1 - np.sum((g.NDVI-g.pred)**2)/np.sum((g.NDVI-g.NDVI.mean())**2)
    print(f"   {c:13s} CV R2 = {r2c:.3f}  (n={len(g)})")

# refit best model on all data, regenerate the dense gap-filled prediction
hgb.fit(Xr, train["NDVI"])
Xall = pd.concat([m[FEATS], pd.get_dummies(m["crop"])], axis=1).reindex(columns=Xr.columns, fill_value=0)
merged["NDVI_s1pred"] = pd.Series(hgb.predict(Xall), index=m.index)  # index-aligned
print("Updated merged['NDVI_s1pred'] with the improved model.")

In [ ]:

# ---- CELL 7: plot gap-filled curve for one example field -------------
fid0 = s2_df["fid"].value_counts().index[0]
g2 = s2_df[s2_df.fid == fid0]; gm = merged[merged.fid == fid0]
plt.figure(figsize=(10,4))
plt.scatter(g2["date"], g2["NDVI"], c="green", label="Sentinel-2 NDVI (clear)", zorder=3)
plt.plot(gm["date"], gm["NDVI_s1pred"], "--", c="orange", label="S1-predicted NDVI (gap-fill)")
plt.title(f"CropPulse gap-filled NDVI - field {fid0} ({gm['crop'].iloc[0]}), {YEAR}")
plt.ylabel("NDVI"); plt.legend(); plt.tight_layout(); plt.savefig("croppulse_gapfill.png", dpi=130)
plt.show()
print("Saved croppulse_gapfill.png")


In [ ]:
# ---- CELL 8 (Phase 2a): per-field phenology SOS/POS/EOS ---------------
from scipy.signal import savgol_filter

GRID = pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31", freq="5D")

def fused_curve(fid):
    # fuse S1-predicted NDVI (dense) with real clear-sky S2 NDVI, resample, smooth
    a = merged.loc[merged.fid==fid, ["date","NDVI_s1pred"]].rename(columns={"NDVI_s1pred":"ndvi"})
    b = s2_df.loc[s2_df.fid==fid, ["date","NDVI"]].rename(columns={"NDVI":"ndvi"})
    s = pd.concat([a, b]).sort_values("date").set_index("date")["ndvi"]
    s = s[~s.index.duplicated(keep="last")].clip(0, 1)
    s = s.reindex(s.index.union(GRID)).interpolate("time").reindex(GRID).bfill().ffill()
    if s.notna().sum() >= 7:
        s = pd.Series(savgol_filter(s.values, 7, 2), index=GRID).clip(0, 1)
    return s

def phenology(s):
    v = s.values; vmin, vmax = float(np.min(v)), float(np.max(v)); amp = vmax - vmin
    pos = int(np.argmax(v)); thr = vmin + 0.5*amp
    sos = next((i for i in range(pos, 0, -1) if v[i] < thr), 0)
    eos = next((i for i in range(pos, len(v)) if v[i] < thr), len(v)-1)
    doy = lambda i: int(GRID[i].dayofyear)
    return dict(SOS_doy=doy(sos), POS_doy=doy(pos), EOS_doy=doy(eos),
                LOS_days=doy(eos)-doy(sos), peak_NDVI=round(vmax, 3))

curves, rows = {}, []
for fid in s2_df["fid"].unique():
    crop = s2_df.loc[s2_df.fid==fid, "crop"].iloc[0]
    s = fused_curve(fid); curves[fid] = (crop, s)
    rows.append(dict(fid=fid, crop=crop, **phenology(s)))
phen = pd.DataFrame(rows)
print(phen.groupby("crop")[["SOS_doy","POS_doy","EOS_doy","LOS_days","peak_NDVI"]].mean().round(1))

# mean phenological curve per crop -> all 7 crops
palette = {"winter_wheat":"#2E7D32","maize":"#E08A1E","winter_barley":"#8E44AD",
           "winter_rape":"#F1C40F","sugar_beet":"#C0392B","grassland":"#16A085",
           "potatoes":"#2980B9"}
plt.figure(figsize=(11,5))
for crop in sorted({c for c,_ in curves.values()}):
    M = pd.DataFrame({fid: s for fid,(c,s) in curves.items() if c==crop})
    if M.empty: continue
    plt.plot(GRID, M.mean(axis=1), lw=2, color=palette.get(crop), label=crop.replace("_"," "))
plt.title(f"CropPulse - mean gap-filled NDVI phenology by crop, Hildesheim {YEAR}")
plt.ylabel("NDVI (gap-filled)"); plt.ylim(0,1); plt.legend(ncol=2); plt.tight_layout()
plt.savefig("croppulse_phenology.png", dpi=130); plt.show()
print("Saved croppulse_phenology.png")


In [ ]:
# ---- CELL 9 (Phase 2c): crop classification wheat vs maize -----------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# features = gap-filled NDVI sampled on the 5-day grid (one column per date)
Xc = pd.DataFrame({fid: s.values for fid,(c,s) in curves.items()}).T
Xc.columns = [d.strftime("%m-%d") for d in GRID]
yc = pd.Series({fid: c for fid,(c,s) in curves.items()}).loc[Xc.index]

clf = RandomForestClassifier(n_estimators=400, random_state=42)
cv  = StratifiedKFold(5, shuffle=True, random_state=42)
pred = cross_val_predict(clf, Xc, yc, cv=cv)
acc = accuracy_score(yc, pred)
print(f"Crop classification (wheat vs maize)  5-fold CV accuracy = {acc:.3f}  (n={len(yc)})")
print("Confusion matrix [rows=true, cols=pred], classes:", sorted(yc.unique()))
print(confusion_matrix(yc, pred, labels=sorted(yc.unique())))
print(classification_report(yc, pred))

# when in the season do the crops separate? -> feature importance over time
clf.fit(Xc, yc)
imp = pd.Series(clf.feature_importances_, index=GRID)
plt.figure(figsize=(10,3))
plt.plot(GRID, imp, color="#444"); plt.fill_between(GRID, 0, imp, color="#2E7D32", alpha=0.2)
plt.title("CropPulse — when wheat & maize separate (RF feature importance over season)")
plt.ylabel("importance"); plt.tight_layout()
plt.savefig("croppulse_importance.png", dpi=130); plt.show()
print("Saved croppulse_importance.png")

In [ ]:
# ---- CELL 10 (Phase 2b): drought / water-stress monitor --------------
# Multi-year growing-season (Apr-Sep) anomalies: ERA5 rainfall + S2 cropland NDVI.
# Negative rainfall AND negative NDVI = drought. Should flag 2018 & 2022.
YEARS = list(range(2018, 2024))
GS = (4, 9)  # Apr..Sep

def gs_precip(y):
    coll = (ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
            .filter(ee.Filter.calendarRange(y, y, "year"))
            .filter(ee.Filter.calendarRange(GS[0], GS[1], "month"))
            .select("total_precipitation_sum"))
    img = coll.sum().multiply(1000)  # m -> mm
    return ee.Number(img.reduceRegion(ee.Reducer.mean(), AOI, 11132)
                        .get("total_precipitation_sum"))

def gs_ndvi(y):
    s, e = f"{y}-{GS[0]:02d}-01", f"{y}-{GS[1]+1:02d}-01"
    coll = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(AOI).filterDate(s, e)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
            .linkCollection(csplus, ["cs_cdf"]).map(mask_s2)
            .map(lambda im: im.normalizedDifference(["B8","B4"]).rename("NDVI")))
    return ee.Number(coll.mean().reduceRegion(ee.Reducer.mean(),
                     fields_all.geometry(), 20, maxPixels=1e9, bestEffort=True).get("NDVI"))

print("Pulling multi-year data (a few minutes)...")
dd = pd.DataFrame({"precip_mm": {y: gs_precip(y).getInfo() for y in YEARS},
                   "ndvi":      {y: gs_ndvi(y).getInfo()  for y in YEARS}})
dd["precip_anom_%"] = (100*(dd.precip_mm/dd.precip_mm.mean()-1)).round(1)
dd["ndvi_anom"]     = (dd.ndvi - dd.ndvi.mean()).round(3)
dd["drought_flag"]  = (dd["precip_anom_%"] < 0) & (dd["ndvi_anom"] < 0)
print(dd.round(3))

fig, ax1 = plt.subplots(figsize=(9,4))
ax1.bar(dd.index, dd["precip_anom_%"],
        color=["#C0392B" if f else "#3498DB" for f in dd.drought_flag], alpha=.75)
ax1.axhline(0, color="k", lw=.8); ax1.set_ylabel("Apr–Sep rainfall anomaly (%)")
ax2 = ax1.twinx(); ax2.plot(dd.index, dd["ndvi_anom"], "o-", color="#2E7D32", lw=2)
ax2.set_ylabel("cropland NDVI anomaly"); ax2.axhline(0, color="#2E7D32", lw=.5, ls=":")
ax1.set_title("CropPulse Drought Monitor — Hildesheim growing-season anomalies")
plt.tight_layout(); plt.savefig("croppulse_drought.png", dpi=130); plt.show()
print("Drought years flagged:", list(dd.index[dd.drought_flag]))

In [ ]:
# ---- CELL 11 (Phase 2d): yield proxy via season-integrated NDVI (iNDVI) ----
# iNDVI = accumulated greenness over the growing season ~ accumulated biomass.
# This is a well-established RELATIVE yield proxy (not calibrated t/ha). We compute
# it per field (2021), then validate a yearly AOI iNDVI against official yield stats.
BASELINE = 0.2          # bare-soil NDVI; integrate greenness above this
DT = 5                  # GRID spacing in days (matches the 5-day GRID)
WIN = {"winter_wheat":(60,220),"winter_barley":(60,200),"winter_rape":(60,210),
       "sugar_beet":(120,300),"potatoes":(120,280),"maize":(120,300),
       "grassland":(1,366)}  # active windows (DOY)

def indvi(s, lo, hi):
    v = np.clip(s.values - BASELINE, 0, None)
    m = (GRID.dayofyear >= lo) & (GRID.dayofyear <= hi)
    return float(np.sum(v[m]) * DT)     # units: NDVI-days

rows = []
for fid,(crop,s) in curves.items():
    lo,hi = WIN.get(crop, (1, 366))
    rows.append(dict(fid=fid, crop=crop, iNDVI=round(indvi(s,lo,hi),1),
                     peak=round(float(s.max()),3)))
yld = pd.DataFrame(rows)
print("Per-field relative yield proxy (2021):")
print(yld.groupby("crop")[["iNDVI","peak"]].agg(["mean","std"]).round(1))

plt.figure(figsize=(7,4))
palette = {"winter_wheat":"#2E7D32","maize":"#E08A1E","winter_barley":"#8E44AD",
           "winter_rape":"#F1C40F","sugar_beet":"#C0392B","grassland":"#16A085",
           "potatoes":"#2980B9"}
for crop in sorted(yld["crop"].unique()):
    plt.hist(yld[yld.crop==crop]["iNDVI"], bins=12, alpha=.45,
             color=palette.get(crop), label=crop.replace("_"," "))
plt.xlabel("season-integrated NDVI (NDVI-days) ~ relative biomass/yield")
plt.ylabel("fields"); plt.legend()
plt.title("CropPulse — relative yield proxy by crop, 2021")
plt.tight_layout(); plt.savefig("croppulse_yieldproxy.png", dpi=130); plt.show()

# ---- multi-year validation: AOI growing-season NDVI vs official wheat yield ----
# Fill with REAL t/ha for the AOI region. Sources:
#  - Destatis table 41241-0005 (national / Bundesland Niedersachsen), or
#  - open per-district dataset (Nature Sci Data 2024): winter wheat by Landkreis
#    incl. Hildesheim, 1979-2021. (For 2022-2023 use Destatis.)
OFFICIAL_YIELD = {      # <-- REPLACE None with real winter-wheat t/ha
    2018: None,   # 2018 = drought year -> expect LOW
    2019: None,
    2020: None,
    2021: None,
    2022: None,   # 2022 = drought year -> expect LOW
    2023: None,
}
if 'dd' in globals() and all(v is not None for v in OFFICIAL_YIELD.values()):
    val = pd.DataFrame({"sat_ndvi": dd["ndvi"],
                        "yield": pd.Series(OFFICIAL_YIELD)}).dropna()
    r = val["sat_ndvi"].corr(val["yield"])
    print(f"\nSatellite NDVI vs official wheat yield: r = {r:.2f} (n={len(val)})")
    plt.figure(figsize=(5,5))
    plt.scatter(val["sat_ndvi"], val["yield"], c="#2E7D32")
    for y,row in val.iterrows(): plt.annotate(str(y),(row.sat_ndvi,row["yield"]))
    plt.xlabel("growing-season AOI NDVI"); plt.ylabel("official wheat yield (t/ha)")
    plt.title(f"CropPulse yield validation  r={r:.2f}")
    plt.tight_layout(); plt.savefig("croppulse_yieldval.png", dpi=130); plt.show()
else:
    print("\n[validation idle] Fill OFFICIAL_YIELD with real t/ha to activate it.")

In [ ]:
# ---- CELL 13 (step 4): realistic multi-crop classification ----------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

Xc = pd.DataFrame({fid: s.values for fid,(c,s) in curves.items()}).T
Xc.columns = [d.strftime("%m-%d") for d in GRID]
yc = pd.Series({fid: c for fid,(c,s) in curves.items()}).loc[Xc.index]
print("classes & counts:\n", yc.value_counts(), "\n")

clf = RandomForestClassifier(n_estimators=500, random_state=42, class_weight="balanced")
cv  = StratifiedKFold(5, shuffle=True, random_state=42)
pred = cross_val_predict(clf, Xc, yc, cv=cv)
acc = accuracy_score(yc, pred)
labels = sorted(yc.unique())
print(f"Multi-crop 5-fold CV accuracy = {acc:.3f} (n={len(yc)})")
print("Confusion matrix [rows=true, cols=pred]:", labels)
print(confusion_matrix(yc, pred, labels=labels))
print(classification_report(yc, pred))

cm = confusion_matrix(yc, pred, labels=labels)
plt.figure(figsize=(6.5,5.5)); plt.imshow(cm, cmap="Greens")
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)
for a in range(len(labels)):
    for b in range(len(labels)):
        plt.text(b, a, cm[a,b], ha="center", va="center",
                 color="white" if cm[a,b] > cm.max()/2 else "black")
plt.xlabel("predicted"); plt.ylabel("true")
plt.title(f"CropPulse - multi-crop confusion (acc={acc:.2f})")
plt.tight_layout(); plt.savefig("croppulse_confusion.png", dpi=130); plt.show()


## CELL 13b - Spatial cross-validation

The classifier above uses a *random* 5-fold split, so neighbouring fields land in
both train and test. That leaks spatial autocorrelation and inflates accuracy.
The cell below holds out whole **geographic blocks** (KMeans on field centroids),
so the model is always tested on ground it never trained on. Report this number.

In [ ]:
# ---- CELL 13b (step 5): SPATIAL cross-validation for the crop classifier ----
# Run AFTER Cell 13 (needs Xc, yc, acc, labels) and after Cell 2 (needs fields_all).
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedGroupKFold

# 1) field centroids (lon/lat) from the GEE FeatureCollection, keyed by fid
cent  = fields_all.map(lambda ft: ft.setGeometry(ft.geometry().centroid(1)))
_info = cent.getInfo()["features"]
coords = (pd.DataFrame([{"fid": f["properties"]["fid"],
                         "lon": f["geometry"]["coordinates"][0],
                         "lat": f["geometry"]["coordinates"][1]} for f in _info])
            .set_index("fid")
            .reindex(Xc.index))                       # align to classifier rows
assert coords["lon"].notna().all(), "some fields have no coordinates -- check fid match"

# 2) group neighbouring fields into spatial blocks
N_BLOCKS = 10
blocks = KMeans(n_clusters=N_BLOCKS, random_state=42, n_init=10)\
            .fit_predict(coords[["lon", "lat"]].values)

# 3) same classifier as Cell 13, but tested on geography it never saw
clf_sp = RandomForestClassifier(n_estimators=500, random_state=42, class_weight="balanced")
sgkf   = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
pred_sp = cross_val_predict(clf_sp, Xc, yc, cv=sgkf, groups=blocks)
acc_sp  = accuracy_score(yc, pred_sp)

print(f"Random  5-fold accuracy = {acc:.3f}   (optimistic - leaks autocorrelation)")
print(f"Spatial 5-fold accuracy = {acc_sp:.3f}   (defensible - the number to report)")
print(f"Optimism gap            = {acc - acc_sp:+.3f}\n")
print(classification_report(yc, pred_sp))

# 4) spatial-CV confusion matrix
cm_sp = confusion_matrix(yc, pred_sp, labels=labels)
plt.figure(figsize=(6.5, 5.5)); plt.imshow(cm_sp, cmap="Greens")
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)
for a in range(len(labels)):
    for b in range(len(labels)):
        plt.text(b, a, cm_sp[a, b], ha="center", va="center",
                 color="white" if cm_sp[a, b] > cm_sp.max()/2 else "black")
plt.xlabel("predicted"); plt.ylabel("true")
plt.title(f"CropPulse - SPATIAL CV confusion (acc={acc_sp:.2f})")
plt.tight_layout(); plt.savefig("croppulse_confusion_spatial.png", dpi=130); plt.show()


In [ ]:
# correctly-titled multi-crop feature importance over the season
clf.fit(Xc, yc)
imp = pd.Series(clf.feature_importances_, index=GRID)
plt.figure(figsize=(10,3))
plt.plot(GRID, imp, color="#444"); plt.fill_between(GRID, 0, imp, color="#2E7D32", alpha=0.2)
plt.title("CropPulse — when the 7 crops separate (RF feature importance over season)")
plt.ylabel("importance"); plt.tight_layout()
plt.savefig("croppulse_importance.png", dpi=130); plt.show()